# 1 — Build the virtual Icechunk store (ARCO-ERA5 single levels)

Replaces `era5-sl-icechunk.ipynb`. Same idea — VirtualiZarr over the public ARCO-ERA5 NetCDF-3
files, chunk manifests committed to Icechunk on your S3 — with the nine issues from
`CODE_REVIEW.md` fixed. The two that change results:

1. **Per-file int16 packing.** Each daily ARCO file has its own `scale_factor`/`add_offset`
   (5 sampled days of 2020 → 5 distinct pairs, both variables). `xr.concat(...,
   combine_attrs='override')` keeps only the first file's pair, so every later day decoded with
   January's packing — up to **≈11 K** of error on `t2m`, silently. Step 3 moves the packing out
   of the array attributes and into per-timestep coordinates, which *do* concatenate correctly.
2. **`total_precipitation` was missing.** It lives at the same path.

Smaller fixes, applied throughout: `Path.home()` instead of `os.environ['HOME']`; no
unconditional recursive delete; `open_or_create` + append; virtual-chunk credentials re-attached
on every open; parallel virtualisation; one commit per month; `join='exact'` so a grid mismatch
raises instead of being absorbed.

In [ ]:
import os, warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

import icechunk
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import NetCDF3Parser
from obspec_utils.registry import ObjectStoreRegistry
from obstore.store import HTTPStore

warnings.filterwarnings("ignore", category=UserWarning)

## Configuration

`total_precipitation` is included. Note the bucket-qualified hostname
(`gcp-public-data-arco-era5.storage.googleapis.com`) — it works anonymously and is friendlier to
restrictive network policies than the generic `storage.googleapis.com`.

In [ ]:
BUCKET_HOST = "https://gcp-public-data-arco-era5.storage.googleapis.com"
CHUNK_ROOT  = BUCKET_HOST + "/"          # registered as the virtual-chunk container prefix

VARIABLES = {                            # ARCO name -> short name inside the NetCDF file
    "2m_temperature":      "t2m",
    "total_precipitation": "tp",
}

DATE_START, DATE_END = "2020-01-01", "2020-01-31"
dates = pd.date_range(DATE_START, DATE_END, freq="D")

def source_url(date, variable):
    return (f"{BUCKET_HOST}/raw/date-variable-single_level/"
            f"{date.year}/{date.month:02d}/{date.day:02d}/{variable}/surface.nc")

# ---- Icechunk target. Path.home() works on Windows; os.environ['HOME'] does not.
dotenv_path = Path.home() / "dotenv" / "protocoast.env"
if dotenv_path.exists():
    from dotenv import load_dotenv
    load_dotenv(dotenv_path, override=True)

STORAGE_ENDPOINT = os.environ.get("ENDPOINT_URL")
BUCKET, STORE_NAME = "protocoast-data", "era5-sl-icechunk-v1"
PREFIX = f"icechunk/{STORE_NAME}"

print(f"{len(dates)} days x {len(VARIABLES)} variables = {len(dates)*len(VARIABLES)} files")
print(f"target: s3://{BUCKET}/{PREFIX}  endpoint={STORAGE_ENDPOINT}")

## Step 1 — inspect one source file

Confirm the shape and, importantly, the packing attributes.

In [ ]:
import fsspec

with fsspec.open(source_url(dates[0], "total_precipitation"), "rb") as f:
    sample = xr.open_dataset(f)
print(sample)
print("\npacking:", {k: sample["tp"].encoding.get(k) for k in ("scale_factor", "add_offset")})

## Step 2 — registry and repository

`authorize_virtual_chunk_access` must be passed **every** time the repository object is
constructed, not only at creation. Omit it and the repo opens fine, then fails later when a
virtual chunk is actually fetched.

In [ ]:
registry = ObjectStoreRegistry({CHUNK_ROOT: HTTPStore.from_url(BUCKET_HOST)})
parser   = NetCDF3Parser()

storage = icechunk.s3_storage(bucket=BUCKET, prefix=PREFIX, from_env=True,
                              endpoint_url=STORAGE_ENDPOINT, region="not-used",
                              force_path_style=True)

repo_config = icechunk.RepositoryConfig.default()
repo_config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(url_prefix=CHUNK_ROOT, store=icechunk.http_store()))
credentials = icechunk.containers_credentials({CHUNK_ROOT: None})   # anonymous HTTPS

def open_repo(create=False):
    opener = icechunk.Repository.open_or_create if create else icechunk.Repository.open
    return opener(storage, repo_config, authorize_virtual_chunk_access=credentials)

print("configured")

## Step 3 — the packing fix

A Zarr array can carry exactly one `scale_factor`/`add_offset`, but every daily file has its own.
So: strip the pair from the array attributes (xarray then does no CF decoding at all) and store it
as two coordinate variables along `time`, which concatenate per timestep like any other data.
`apply_packing` decodes with them on read.

The chunk manifests are untouched, so the store stays virtual and zero-copy.

In [ ]:
SF_SUFFIX, AO_SUFFIX = "_scale_factor", "_add_offset"

def read_packing(vds, short):
    enc = {**vds[short].attrs, **getattr(vds[short], "encoding", {})}
    return (float(np.ravel(enc.get("scale_factor", 1.0))[0]),
            float(np.ravel(enc.get("add_offset", 0.0))[0]))

def externalise_packing(vds, short):
    sf, ao = read_packing(vds, short)
    out = vds.copy()
    for holder in (out[short].attrs, getattr(out[short], "encoding", {})):
        holder.pop("scale_factor", None)
        holder.pop("add_offset", None)
    nt = out.sizes["time"]
    return out.assign_coords({
        short + SF_SUFFIX: ("time", np.full(nt, sf, dtype="float64")),
        short + AO_SUFFIX: ("time", np.full(nt, ao, dtype="float64")),
    })

def apply_packing(ds):
    out = ds
    for short in list(out.data_vars):
        sf, ao = short + SF_SUFFIX, short + AO_SUFFIX
        if sf in out.coords and ao in out.coords:
            attrs = ds[short].attrs
            out = out.assign({short: out[short] * out[sf] + out[ao]})
            out[short].attrs = attrs
    return out.drop_vars([c for c in out.coords
                          if c.endswith((SF_SUFFIX, AO_SUFFIX))], errors="ignore")

## Step 4 — virtualise and commit, one month at a time

Parallel over days, one commit per calendar month, and `join='exact'` so a grid mismatch between
variables raises rather than being silently absorbed by `compat='override'`.

Nothing is deleted implicitly. Set `APPEND = True` to extend an existing store along `time`.

In [ ]:
APPEND = False
MAX_WORKERS = 8
packing_log = {}

def virtualize_day(date):
    parts = []
    for var, short in VARIABLES.items():
        vds = open_virtual_dataset(source_url(date, var), parser=parser, registry=registry,
                                   loadable_variables=["time", "latitude", "longitude"])
        packing_log.setdefault(short, []).append(read_packing(vds, short))
        parts.append(externalise_packing(vds, short))
    return xr.merge(parts, join="exact", combine_attrs="drop_conflicts")

repo = open_repo(create=not APPEND)
months = dates.groupby(dates.to_period("M"))

for i, (period, month_dates) in enumerate(months.items()):
    with ThreadPoolExecutor(MAX_WORKERS) as ex:
        daily = list(ex.map(virtualize_day, month_dates))
    vds = xr.concat(daily, dim="time", coords="minimal",
                    compat="override", combine_attrs="drop_conflicts")
    session = repo.writable_session("main")
    writer  = getattr(vds, "vz", None) or vds.virtualize      # virtualizarr >=2 / <2
    writer.to_icechunk(session.store, **({"append_dim": "time"} if (APPEND or i) else {}))
    snap = session.commit(f"ERA5 single-levels {period} "
                          f"({len(VARIABLES)} vars, {len(month_dates)*24} steps)")
    print(f"committed {period}: {snap}")

for short, rows in packing_log.items():
    pairs = {(round(sf, 12), round(ao, 12)) for sf, ao in rows}
    if len(pairs) > 1:
        print(f"note: {short!r} had {len(pairs)} distinct packings across {len(rows)} files "
              f"- externalised to {short}{SF_SUFFIX}/{short}{AO_SUFFIX}")

## Step 5 — verify against the source

The check that matters: pick a day that is **not** the first in the store and compare the store
against the original file. With the old code this differed by several kelvin.

In [ ]:
def open_era5(branch="main"):
    ds = xr.open_zarr(open_repo().readonly_session(branch).store,
                      consolidated=False, chunks={})
    return apply_packing(ds)

ds = open_era5()
print(ds)

check_time = str(dates[min(6, len(dates)-1)].date()) + "T12:00"
with fsspec.open(source_url(pd.Timestamp(check_time), "2m_temperature"), "rb") as f:
    direct = xr.open_dataset(f)

a = float(ds["t2m"].sel(time=check_time).mean())
b = float(direct["t2m"].sel(time=check_time).mean())
print(f"{check_time}  store {a:.4f} K   source {b:.4f} K   diff {abs(a-b):.2e} K")
assert abs(a - b) < 0.01, "packing mismatch - see CODE_REVIEW.md #1"

## Step 6 — quick look

Units: K → °C, and ERA5 `tp` is an hourly **accumulation** in metres, so m → mm.

In [ ]:
import matplotlib.pyplot as plt

t2m = ds["t2m"].isel(time=0) - 273.15
tp  = ds["tp"].sel(time=slice(str(dates[0].date()), str(dates[0].date()))).sum("time") * 1000.0

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), constrained_layout=True)
t2m.sel(latitude=slice(48, 36), longitude=slice(6, 20)).plot(
    ax=axes[0], cmap="RdYlBu_r", cbar_kwargs={"label": "2 m temperature [degC]"})
tp.sel(latitude=slice(48, 36), longitude=slice(6, 20)).plot(
    ax=axes[1], cmap="BuPu", cbar_kwargs={"label": "daily total precipitation [mm]"})
for ax in axes:
    ax.set_title("")
fig